In [9]:
from pathlib import Path
import sys


def encontrar_raiz_projeto() -> Path:
    pasta_atual = Path.cwd().resolve()

    for pasta in [pasta_atual, *pasta_atual.parents]:
        if (pasta / "src").exists() and (pasta / "configs").exists():
            return pasta

    raise FileNotFoundError(
        "Não foi possível localizar a raiz do projeto."
    )


ROOT_DIR = encontrar_raiz_projeto()

if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))


from src.utils import env
from src.utils import storage


env.carregar_dotenv()

print(f"Raiz do projeto: {ROOT_DIR}")

Raiz do projeto: C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador


In [10]:
COLUNAS_EDA = [
    "nu_notific",
    "dt_notific",
    "dt_sin_pri",
    "sem_pri",
    "cs_sexo",
    "nu_idade_n",
    "tp_idade",
    "cs_gestant",
    "ave_suino",
    "febre",
    "tosse",
    "garganta",
    "dispneia",
    "diarreia",
    "vomito",
    "outro_sin",
    "puerpera",
    "cardiopati",
    "hematologi",
    "sind_down",
    "hepatica",
    "asma",
    "diabetes",
    "neurologic",
    "pneumopati",
    "imunodepre",
    "renal",
    "obesidade",
    "obes_imc",
    "out_morbi",
    "tabag",
    "vacina",
    "dt_ut_dose",
    "mae_vac",
    "dt_vac_mae",
    "dt_doseuni",
    "dt_1_dose",
    "dt_2_dose",
    "hospital",
    "dt_interna",
    "evolucao",
    "dt_evoluca",
    "dt_encerra",
    "dt_digita",
    "out_anim",
    "dor_abd",
    "fadiga",
    "perd_olft",
    "perd_pala",
    "vacina_cov",
    "dose_1_cov",
    "dose_2_cov",
    "dose_ref",
    "dose_2ref",
    "dose_adic",
    "dos_re_bi"
]

In [ ]:
caminho_parquet = storage.get_parquet()

print(f"Arquivo: {caminho_parquet}")
print(f"Existe: {caminho_parquet.exists()}")
print(
    f"Tamanho: "
    f"{caminho_parquet.stat().st_size / 1024**2:,.2f} MB"
)

In [ ]:
import pyarrow.parquet as pq


arquivo_parquet = pq.ParquetFile(caminho_parquet)

colunas_disponiveis = arquivo_parquet.schema_arrow.names

print(
    f"Linhas registradas no Parquet: "
    f"{arquivo_parquet.metadata.num_rows:,}"
)

print(
    f"Colunas registradas no Parquet: "
    f"{arquivo_parquet.metadata.num_columns:,}"
)

print(
    f"Row groups: "
    f"{arquivo_parquet.metadata.num_row_groups:,}"
)

In [ ]:
for coluna in COLUNAS_EDA:
    print(coluna)

In [ ]:
colunas_disponiveis = set(metadados["colunas"])

colunas_ausentes = [
    coluna
    for coluna in COLUNAS_EDA
    if coluna not in colunas_disponiveis
]

if colunas_ausentes:
    print("Colunas que não foram encontradas:")

    for coluna in colunas_ausentes:
        print(f"- {coluna}")
else:
    print(
        "Todas as colunas solicitadas existem no Parquet."
    )

NameError: name 'metadados' is not defined

In [ ]:
df = storage.read_srag(
    columns=COLUNAS_EDA,
)

print(f"Linhas carregadas: {df.shape[0\]:,}")
print(f"Colunas carregadas: {df.shape[1\]:,}")

display(df.head())

In [ ]:
print(f"Linhas: {df.shape,}")
print(f"Colunas: {df.shape,}")

In [ ]:
tipos = (
    df.dtypes
    .astype(str)
    .rename("tipo")
    .reset_index()
    .rename(columns={"index": "coluna"})
)

display(tipos)

In [ ]:
df.info()

In [ ]:
memoria_mb = (
    df.memory_usage(deep=True).sum()
    / 1024**2
)

print(f"Memória total utilizada: {memoria_mb:,.2f} MB")

In [ ]:
memoria_colunas = (
    df.memory_usage(deep=True)
    .div(1024**2)
    .rename("memoria_mb")
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={"index": "coluna"})
)

display(memoria_colunas)

In [ ]:
ausencias = (
    df.isna()
    .sum()
    .rename("quantidade_ausente")
    .to_frame()
)

ausencias["percentual_ausente"] = (
    ausencias["quantidade_ausente"]
    / len(df)
    * 100
)

ausencias = ausencias.sort_values(
    "percentual_ausente",
    ascending=False,
)

display(ausencias)

In [ ]:
cardinalidade = (
    df.nunique(dropna=False)
    .rename("quantidade_valores_distintos")
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={"index": "coluna"})
)

display(cardinalidade)

In [ ]:
# df["DT_NOTIFIC"] = pd.to_datetime(
#     df["DT_NOTIFIC"],
#     errors="coerce",
# )

# df["ANO_NOTIFICACAO"] = (
#     df["DT_NOTIFIC"].dt.year
# )

# display(
#     df["ANO_NOTIFICACAO"]
#     .value_counts(dropna=False)
#     .sort_index()
# )

df["ANO"].value_counts(
    dropna=False
).sort_index()

In [ ]:
caminho_saida = (
RAIZ_PROJETO
/ "outputs"
/ "tables"
/ "validacao_colunas_eda.csv"
)
 
tipos.to_csv(
caminho_saida,
index=False,
encoding="utf-8-sig",
)
 
print(f"Arquivo salvo em: {caminho_saida}")

In [ ]:
caminho_ausencias = (
    RAIZ_PROJETO
    / "outputs"
    / "tables"
    / "ausencias_primeira_passagem.csv"
)

ausencias.to_csv(
    caminho_ausencias,
    encoding="utf-8-sig",
)

print(f"Arquivo salvo em: {caminho_ausencias}")